# 第七课｜第一个 RTL 神经元

寄存器传输级（Register-Transfer Level, RTL）描述寄存器之间的计算和更新。本课实现积分发放（Integrate-and-Fire, IF）规则：累加输入，比较阈值，发放时复位。它由加法器、比较器、选择器和两个状态寄存器组成。


## 1. 概念账本

**已经知道：** register、clock edge、RTL module/port，并见过一个简单 `always_ff`。

**今天学习：** combinational path、`always_comb`，以及它怎样与 sequential `always_ff` 配合。

**只预告：** testbench / waveform 留到下一课。


## 2. 每个时钟沿的更新规则

1. 用旧 `membrane_v` 和输入求 `candidate`。
2. 用 `candidate >= threshold` 得到 `spike_next`。
3. 若满足阈值，`next_v=reset_value`；否则 `next_v=candidate`。
4. 上升沿复位有效时保存 `reset_value`，并清零 `spike`；否则保存 `next_v` 和 `spike_next`。

信号是 8 位有符号数。主实验的和位于 −128 到 127；后面的溢出实验单独展示保留低八位的结果。这个 IF 例子只有积分、阈值和复位，漏电项需要另外的状态更新计算。


## 3. 先用 Python oracle 重放规则

先预测 threshold=4、输入 `[1,1,1,1,2,2]` 的 spike cycle，再运行。


In [ ]:
def tutorial_if_step(state, input_current, threshold, reset_value=0):
    candidate = state + input_current
    spike = candidate >= threshold
    next_state = reset_value if spike else candidate
    return next_state, spike, candidate

state = 0
for cycle, current in enumerate([1, 1, 1, 1, 2, 2]):
    next_state, spike, candidate = tutorial_if_step(state, current, 4)
    print(f'cycle={cycle}: state={state}, input={current}, candidate={candidate}, spike={spike}, next={next_state}')
    state = next_state


## 4. 结构图

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 840 300" role="img" aria-label="l07-flow" style="width:100%;height:auto">
<defs><marker id="l07-flow" markerWidth="8" markerHeight="8" refX="7" refY="4" orient="auto"><path d="M0,0 L8,4 L0,8 Z" fill="#506070"/></marker></defs>
<rect x="20" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="95.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">membrane_v</text>
<rect x="225" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="300.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">adder</text>
<rect x="430" y="100" width="150" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="505.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">comparator / mux</text>
<rect x="635" y="100" width="180" height="60" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="725.0" y="135.0" text-anchor="middle" font-family="sans-serif" font-size="15">state + spike regs</text>
<rect x="225" y="15" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="300.0" y="42.5" text-anchor="middle" font-family="sans-serif" font-size="15">input_current</text>
<rect x="430" y="15" width="150" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="505.0" y="42.5" text-anchor="middle" font-family="sans-serif" font-size="15">threshold / reset</text>
<rect x="635" y="215" width="180" height="45" rx="6" fill="#e6f4ea" stroke="#506070"/><text x="725.0" y="242.5" text-anchor="middle" font-family="sans-serif" font-size="15">clk / rst_n</text>
<path d="M170 130 H225" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M375 130 H430" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="402" y="118" text-anchor="middle" font-family="sans-serif" font-size="14">candidate</text>
<path d="M580 130 H635" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="607" y="185" text-anchor="middle" font-family="sans-serif" font-size="14">next_v / spike_next</text>
<path d="M300 60 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M505 60 V100" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M725 215 V160" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<path d="M815 130 H830 V285 H95 V160" fill="none" stroke="#506070" stroke-width="2" marker-end="url(#l07-flow)"/>
<text x="390" y="278" text-anchor="middle" font-family="sans-serif" font-size="14">stored membrane_v</text>
</svg>


## 5. `always_comb` 与 `always_ff` 怎样配合

二者都是 SystemVerilog 标准关键字。`always_comb` 描述组合计算，自动确定相关读取信号的敏感性，并在仿真开始时执行一次。本例读取状态、输入、阈值和复位值，依次算出 `candidate`、`spike_next`、`next_v`。

组合块使用 `=`：前一条赋值完成后，后一条可以读取刚得到的中间值。例如先得到 `candidate=4`，再比较它是否达到阈值。`next_v` 在 `if` 和 `else` 两条分支上都有赋值，因此无需保存它的历史值。遗漏一条分支会造成锁存行为，自动敏感性不能补齐该赋值。

`always_ff @(posedge clk)` 使用 `<=` 将组合结果写入 `membrane_v` 和 `spike`。同一个边沿上的右侧值在寄存器更新前求得。状态更新后，组合逻辑会根据新状态再次计算，所以边沿前的 `candidate` 与边沿后的 `candidate` 可能不同。


## 6. 完整教学 RTL

```systemverilog
module tutorial_if_neuron (
    input  logic              clk,
    input  logic              rst_n,
    input  logic signed [7:0] input_current,
    input  logic signed [7:0] threshold,
    input  logic signed [7:0] reset_value,
    output logic signed [7:0] membrane_v,
    output logic              spike
);
    logic signed [7:0] candidate;
    logic signed [7:0] next_v;
    logic spike_next;

    always_comb begin
        candidate = membrane_v + input_current;
        spike_next = candidate >= threshold;

        if (spike_next)
            next_v = reset_value;
        else
            next_v = candidate;
    end

    always_ff @(posedge clk) begin
        if (!rst_n) begin
            membrane_v <= reset_value;
            spike <= 1'b0;
        end else begin
            membrane_v <= next_v;
            spike <= spike_next;
        end
    end
endmodule
```


## 7. 从语句读出电路

`candidate` 和 `next_v` 的声明表示八位有符号中间信号；`spike_next` 是一位条件。`candidate = membrane_v + input_current` 对应加法器，`>=` 对应比较器，`if/else` 对应选择器。

`membrane_v` 和 `spike` 是时钟沿更新的输出状态。`1'b0` 是一位二进制零：`1` 指位宽、`b` 指二进制、末尾 `0` 指数值。`8'sd4` 则是八位有符号十进制常量 4。

边沿前若 `membrane_v=3,input_current=1,threshold=4,reset_value=0`，则 `candidate=4,spike_next=1,next_v=0`。该边沿后 `membrane_v=0,spike=1`。输入仍为 1 时，组合块重新计算出 `candidate=1,spike_next=0,next_v=1`；输出 `spike` 仍保持 1，直到下一个上升沿更新。


### 可选实验：8-bit overflow 为什么会让直觉失效？

当前教学 RTL 的 `candidate` 只有 8-bit signed，范围是 `-128..127`。如果数学上得到的结果超出范围，硬件会保留有限位宽的 bit pattern；按 two's-complement 解释时就可能变成负数。

例如先预测 `100 + 50` 在 8-bit signed 中会变成什么，然后运行下面的 Python 小实验。

In [ ]:
def wrap_signed_8(x: int) -> int:
    return ((x + 128) % 256) - 128

mathematical_sum = 100 + 50
wrapped_sum = wrap_signed_8(mathematical_sum)
print('mathematical sum =', mathematical_sum)
print('8-bit signed result =', wrapped_sum)


结果为 `150 → −106`：150 的低八位是 `10010110`，作为有符号补码解释为 −106。若阈值为正，比较可能得到“不发放”。要得到数学意义上的 150，需要加宽运算和存储；若要限制到最大正值，需要显式饱和计算。

若 `reset_value >= threshold` 且输入为 0，复位后的状态本身达到阈值，后续每拍都会发放。这由本例规则直接决定。正式神经元的溢出和参数限制集中记录在项目追踪所指的数值规范中。


## 8. Try It：手算一个 edge

edge 前：`membrane_v=3, input_current=1, threshold=4, reset_value=0`。先写 candidate / spike_next / next_v，再写 edge 后 membrane_v / spike。


## 作业

[第 7 课作业：拆开 combinational path 与 register edge](../../exercises/zh/07_first_rtl_neuron.ipynb)

这份独立作业检查本课的语义理解；课程中的真实 RTL / testbench 实验仍然保留。

## 9. AI Task

让 AI 对照本课五条 contract，逐条指出 RTL 中对应的语句。如果发现不一致，只报告差异，不允许它改 contract。


## 10. Human Check

不用 AI，你应该能指出 combinational path 与 stored state；解释 `always_comb` / `always_ff` 的分工；说明为什么 `candidate=4` 与 edge 后 `membrane_v=0` 可以同时正确；说明为什么本教学模块不能冒充正式 LIF RTL。


## 11. Engineering Handoff

`rtl/learning/tutorial_if_neuron.sv` 只验证“会把已知 contract 映射到 RTL 结构”。正式 `rtl/neuron/lif_neuron_engine.sv` 仍等待 v0 semantics 与 fixed-point contract。


## 12. 项目追踪 Project Trace

- Lesson: `LSN-007`
- Mapping: `RMD-004` teaching precursor
- Formal `MOD-003`: intentionally not created


## 13. Exit Ticket

你能从 contract 画出 combinational path + registers，并从 RTL 指出两部分。下一课不再增加 neuron 规则，只学习怎样验证它。
